# Shallow wormhole test on Tuna-17
Click **Runtime → Run all**. When step 2 shows a **login link**, click it, log in, and come back. Everything else runs by itself; the result downloads at the end. If it says *queue full*, just wait: it retries every minute.

**1. Install and write the scripts** (1–2 minutes)

In [ ]:
!pip install -q qiskit-quantuminspire quantuminspire scipy
open('wormhole_ibm.py', 'w').write('"""\nWormhole-inspired teleportation on a quantum computer (toy model, 7 qubits).\n\nWhat it does\n------------\nTwo small "black holes" L and R (3 qubits each) start entangled in Bell pairs\n(the thermofield-double state at infinite temperature). A message qubit Q is\nthrown into L at time -t, the two sides are coupled once with strength mu\n(the analogue of the negative-energy pulse that holds a wormhole open), and the\nmessage is read off R\'s first qubit at time +t.\n\nDynamics: a chaotic kicked Ising chain, one step = ZZ bonds + X and Z fields.\nThe circuit itself IS the model, so the local simulator is exact.\n\nSuccess measure: average teleportation fidelity over the six Pauli eigenstates.\n    0.50 = no information gets through (random guess)\n    0.667 = best any classical (measure-and-resend) channel can do\n    1.00 = perfect teleportation\nControls: mu = 0 (no coupling: should give 0.50) and mu of the opposite sign.\n\nUsage\n-----\n    pip install qiskit qiskit-aer qiskit-ibm-runtime scipy\n    python wormhole_ibm.py scan        # find the best (steps, mu) on the exact simulator\n    python wormhole_ibm.py simulate    # exact + noisy (IBM Heron noise model) prediction\n    python wormhole_ibm.py hardware    # real IBM QPU; needs your token (see below)\n\nFor hardware, create a free account at quantum.cloud.ibm.com, copy your API key, then\n    export IBM_QUANTUM_TOKEN="your-key"          (Windows: set IBM_QUANTUM_TOKEN=your-key)\n    python wormhole_ibm.py hardware --shots 2000\nIf the session closes while the job queues, collect it later with\n    python wormhole_ibm.py fetch --job <job id>\nThe default run is 18 circuits x 2000 shots: well inside the free 10 minutes.\nOptional: --backend ibm_xxx to pick a machine; otherwise the least busy one is used.\n"""\nimport argparse\nimport json\nimport os\nimport sys\n\nimport numpy as np\nfrom qiskit import QuantumCircuit, transpile\nfrom qiskit.quantum_info import Statevector, partial_trace, DensityMatrix\nfrom scipy.linalg import expm\nfrom scipy.optimize import minimize\n\n# ---------------- model parameters (chosen by `scan`; see README) ----------------\nN = 3            # qubits per side\nHX, HZ = 1.05, 0.5\nDT = 0.25        # time per Floquet step (chaotic model)\nMODEL = \'chaotic\'\nDEFAULTS = {"steps": None, "mu": None}  # filled from scan_result.json if present\n\nPX = np.array([[0, 1], [1, 0]], dtype=complex)\nPY = np.array([[0, -1j], [1j, 0]], dtype=complex)\nPZ = np.array([[1, 0], [0, -1]], dtype=complex)\n\n\n# ---------------- circuit construction ----------------\ndef floquet_step(n):\n    """One Trotter step of the mixed-field Ising chain H = sum ZZ + HX sum X + HZ sum Z."""\n    qc = QuantumCircuit(n)\n    for i in range(n - 1):\n        qc.rzz(2 * DT, i, i + 1)\n    for i in range(n):\n        qc.rx(2 * HX * DT, i)\n        qc.rz(2 * HZ * DT, i)\n    return qc\n\n\ndef evolution(n, steps):\n    qc = QuantumCircuit(n)\n    step = floquet_step(n)\n    for _ in range(steps):\n        qc.compose(step, inplace=True)\n    return qc\n\n\ndef protocol(steps, mu, with_reference=False):\n    """Qubits: [P] Q, L1..LN, R1..RN.  Returns circuit (no measurement).\n    The message is inserted by SWAP(Q, L1); we skip the SWAP gate by letting Q\n    *become* L1 for the forward evolution (old L1 then sits idle)."""\n    off = 1 if with_reference else 0\n    q = off\n    L = list(range(off + 1, off + 1 + N))\n    R = list(range(off + 1 + N, off + 1 + 2 * N))\n    qc = QuantumCircuit(off + 1 + 2 * N)\n    if with_reference:\n        qc.h(0); qc.cx(0, q)                      # P-Q Bell pair (for exact analysis)\n    for l, r in zip(L, R):                        # TFD at infinite temperature\n        qc.h(l); qc.cx(l, r)\n    U = evolution(N, steps)\n    qc.compose(U.inverse(), qubits=L, inplace=True)          # e^{+iH_L t}\n    Lnew = [q] + L[1:]                                       # message now plays L1\n    qc.compose(U, qubits=Lnew, inplace=True)                 # e^{-iH_L t}\n    for l, r in zip(Lnew[1:], R[1:]):                        # coupling e^{i mu V}\n        qc.rzz(-2 * mu / (N - 1), l, r)\n    qc.compose(U, qubits=R, inplace=True)                    # e^{-iH_R t}\n    return qc, q, R[0]\n\n\n# ---------------- exact analysis ----------------\ndef choi_state(steps, mu):\n    qc, _, r1 = protocol(steps, mu, with_reference=True)\n    sv = Statevector(qc)\n    keep = [0, r1]\n    rho = partial_trace(sv, [i for i in range(qc.num_qubits) if i not in keep])\n    # qiskit orders qubits little-endian: reorder to (P, R1)\n    return DensityMatrix(rho).data.reshape(2, 2, 2, 2).transpose(1, 0, 3, 2).reshape(4, 4)\n\n\ndef mutual_info(rho):\n    S = lambda m: -sum(x * np.log2(x) for x in np.linalg.eigvalsh(m) if x > 1e-12)\n    r = rho.reshape(2, 2, 2, 2)\n    return S(np.einsum(\'ajbj->ab\', r)) + S(np.einsum(\'iaib->ab\', r)) - S(rho)\n\n\ndef su2(p):\n    return expm(-1j * (p[0] * PX + p[1] * PY + p[2] * PZ))\n\n\ndef best_decoder(rho):\n    phi = np.array([1, 0, 0, 1], dtype=complex) / np.sqrt(2)\n    def neg(p):\n        W = np.kron(np.eye(2), su2(p))\n        return -np.real(phi.conj() @ W @ rho @ W.conj().T @ phi)\n    res = min((minimize(neg, x0, method=\'Nelder-Mead\', options={\'xatol\': 1e-6, \'fatol\': 1e-9})\n               for x0 in np.random.default_rng(1).normal(size=(10, 3))), key=lambda r: r.fun)\n    return (2 * (-res.fun) + 1) / 3, res.x     # average fidelity, decoder params\n\n\n# ---------------- measurement circuits ----------------\nINPUTS = [\'+z\', \'-z\', \'+x\', \'-x\', \'+y\', \'-y\']\n\n\ndef prep(qc, q, label):\n    if label == \'-z\': qc.x(q)\n    elif label == \'+x\': qc.h(q)\n    elif label == \'-x\': qc.x(q); qc.h(q)\n    elif label == \'+y\': qc.h(q); qc.s(q)\n    elif label == \'-y\': qc.x(q); qc.h(q); qc.s(q)\n\n\ndef rotate_to_z(qc, q, label):\n    """Map the input basis state back to |0>/|1> so a Z measurement checks it."""\n    if label[1] == \'x\': qc.h(q)\n    elif label[1] == \'y\': qc.sdg(q); qc.h(q)\n\n\ndef measurement_circuit(steps, mu, label, decoder):\n    core, q, r1 = protocol(steps, mu)\n    qc = QuantumCircuit(core.num_qubits, 1)\n    prep(qc, q, label)\n    qc.compose(core, inplace=True)\n    qc.unitary(su2(decoder), [r1], label=\'decode\')\n    rotate_to_z(qc, r1, label)\n    qc.measure(r1, 0)\n    qc.name = f"mu={mu:+.3f} in={label}"\n    return qc\n\n\ndef fidelity_from_counts(counts, label):\n    shots = sum(counts.values())\n    want = \'0\' if label[0] == \'+\' else \'1\'\n    return counts.get(want, 0) / shots\n\n\n# ---------------- commands ----------------\ndef cmd_scan(args):\n    best = None\n    rows = []\n    for steps in range(1, 13):\n        for mu in np.linspace(-2 * np.pi, 2 * np.pi, 49):\n            I = mutual_info(choi_state(steps, mu))\n            rows.append((steps, float(mu), float(I)))\n            if best is None or I > best[2]:\n                best = (steps, float(mu), float(I))\n    steps, mu, I = best\n    F, dec = best_decoder(choi_state(steps, mu))\n    Fneg, _ = best_decoder(choi_state(steps, -mu))\n    F0, _ = best_decoder(choi_state(steps, 0.0))\n    out = {"steps": steps, "mu": mu, "I_bits": I, "F_best": F, "F_minus_mu": Fneg,\n           "F_mu0": F0, "decoder": dec.tolist(), "grid": rows}\n    json.dump(out, open(f\'scan_result_{MODEL}.json\', \'w\'), indent=1)\n    print(f"[{MODEL}] best: steps={steps}, mu={mu:+.3f}, I(P:R1)={I:.3f} bits (max 2)")\n    print(f"exact average fidelity: best mu {F:.3f} | opposite sign {Fneg:.3f} | mu=0 {F0:.3f}  (classical limit 0.667)")\n\n\ndef load_settings():\n    f = f\'scan_result_{MODEL}.json\'\n    if not os.path.exists(f):\n        sys.exit(f"run `python wormhole_ibm.py scan --model {MODEL}` first")\n    return json.load(open(f))\n\n\ndef build_batch(s):\n    circs, meta = [], []\n    for mu in (s[\'mu\'], -s[\'mu\'], 0.0):\n        dec = best_decoder(choi_state(s[\'steps\'], mu))[1] if mu != 0 else np.zeros(3)\n        if mu == s[\'mu\']:\n            dec = np.array(s[\'decoder\'])\n        for lab in INPUTS:\n            circs.append(measurement_circuit(s[\'steps\'], mu, lab, dec))\n            meta.append((mu, lab))\n    return circs, meta\n\n\ndef summarise(meta, all_counts, title):\n    res = {}\n    for (mu, lab), c in zip(meta, all_counts):\n        res.setdefault(mu, []).append(fidelity_from_counts(c, lab))\n    best_mu = meta[0][0]\n    print(title)\n    summary = {}\n    for mu, fs in res.items():\n        F = float(np.mean(fs))\n        shots = sum(all_counts[0].values()) * len(fs)\n        err = np.sqrt(F * (1 - F) / shots)\n        summary[f"{mu:+.3f}"] = {"F": F, "stderr": float(err)}\n        tag = ("coupling ON (best setting)" if mu == best_mu else\n               "no coupling (control)" if mu == 0 else "coupling, opposite sign")\n        print(f"  mu={mu:+.3f}  F_avg = {F:.3f} ± {err:.3f}   [{tag}]")\n    print("  (0.500 = nothing gets through, 0.667 = classical limit)")\n    return summary\n\n\ndef cmd_simulate(args):\n    from qiskit_aer import AerSimulator\n    s = load_settings()\n    circs, meta = build_batch(s)\n    ideal = AerSimulator()\n    counts = ideal.run(transpile(circs, ideal), shots=args.shots, seed_simulator=7).result().get_counts()\n    out = {"ideal": summarise(meta, counts, "\\nNoiseless simulator:")}\n    try:\n        from qiskit_ibm_runtime.fake_provider import FakeFez\n        fake = FakeFez()\n        noisy = AerSimulator.from_backend(fake)\n        tc = transpile(circs, fake, optimization_level=3, seed_transpiler=7)\n        print(f"\\n(transpiled for {fake.name}: ~{max(c.count_ops().get(\'cz\', 0) for c in tc)} two-qubit gates per circuit)")\n        counts = noisy.run(tc, shots=args.shots, seed_simulator=7).result().get_counts()\n        out["noisy_" + fake.name] = summarise(meta, counts, f"With {fake.name} noise model:")\n    except Exception as e:\n        print("noisy simulation skipped:", e)\n    json.dump(out, open(f\'simulate_result_{MODEL}.json\', \'w\'), indent=1)\n\n\ndef cmd_hardware(args):\n    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2\n    token = os.environ.get(\'IBM_QUANTUM_TOKEN\')\n    if not token:\n        sys.exit("set IBM_QUANTUM_TOKEN to your IBM Quantum API key first")\n    # Only ever use a free Open Plan instance, so the run can never be billed.\n    service = QiskitRuntimeService(channel=\'ibm_quantum_platform\', token=token, plans_preference=[\'open\'])\n    print(\'instance:\', service.active_instance() if hasattr(service, \'active_instance\') else \'open plan\')\n    backend = service.backend(args.backend) if args.backend else service.least_busy(operational=True, simulator=False)\n    print("using", backend.name)\n    s = load_settings()\n    circs, meta = build_batch(s)\n    tc = transpile(circs, backend, optimization_level=3, seed_transpiler=7)\n    sampler = SamplerV2(mode=backend)\n    sampler.options.dynamical_decoupling.enable = True\n    sampler.options.twirling.enable_gates = True\n    job = sampler.run(tc, shots=args.shots)\n    print("job id:", job.job_id(), "- waiting in queue (can take a while)...")\n    result = job.result()\n    counts = [r.data.c.get_counts() for r in result]\n    summary = summarise(meta, counts, f"\\n{backend.name} (real hardware):")\n    json.dump({"backend": backend.name, "job": job.job_id(), "model": MODEL, "summary": summary,\n               "counts": counts}, open(f\'hardware_result_{MODEL}.json\', \'w\'), indent=1)\n\n\ndef cmd_fetch(args):\n    """Collect a finished hardware job later, e.g. if the session closed while it queued."""\n    from qiskit_ibm_runtime import QiskitRuntimeService\n    token = os.environ.get(\'IBM_QUANTUM_TOKEN\')\n    if not token or not args.job:\n        sys.exit("set IBM_QUANTUM_TOKEN and pass --job <job id>")\n    service = QiskitRuntimeService(channel=\'ibm_quantum_platform\', token=token, plans_preference=[\'open\'])\n    job = service.job(args.job)\n    print("job status:", job.status())\n    result = job.result()\n    s = load_settings()\n    _, meta = build_batch(s)\n    counts = [r.data.c.get_counts() for r in result]\n    name = job.backend().name\n    summary = summarise(meta, counts, f"\\n{name} (real hardware):")\n    json.dump({"backend": name, "job": args.job, "model": MODEL, "summary": summary,\n               "counts": counts}, open(f\'hardware_result_{MODEL}.json\', \'w\'), indent=1)\n\n\nif __name__ == \'__main__\':\n    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    ap.add_argument(\'command\', choices=[\'scan\', \'simulate\', \'hardware\', \'fetch\'])\n    ap.add_argument(\'--shots\', type=int, default=2000)\n    ap.add_argument(\'--backend\', default=None)\n    ap.add_argument(\'--job\', default=None, help=\'job id, for the fetch command\')\n    ap.add_argument(\'--model\', choices=[\'chaotic\'], default=\'chaotic\')\n    a = ap.parse_args()\n    MODEL = a.model\n    {\'scan\': cmd_scan, \'simulate\': cmd_simulate, \'hardware\': cmd_hardware, \'fetch\': cmd_fetch}[a.command](a)\n')
open('qi_shallow.py', 'w').write('"""Shallow wormhole-teleportation test for Quantum Inspire\'s Tuna-17.\n2 qubits per side (5 qubits in all), 4 time steps of 0.75, coupling mu = -pi/4:\nabout 13 two-qubit gates before routing, versus ~89 for the deep version.\nNoise-model prediction: average fidelity ~0.53 with coupling vs 0.50 without."""\nimport json, sys, time\nimport numpy as np\nfrom qiskit import transpile\nimport wormhole_ibm as w\n\nw.MODEL, w.N, w.DT = \'chaotic\', 2, 0.75\nSTEPS, MU, SHOTS = 4, -np.pi / 4, 8000\nWANT = sys.argv[1] if len(sys.argv) > 1 else \'Tuna-17\'\n\n\ndef bit(key):\n    k = str(key).replace(\' \', \'\')\n    return int(k, 16) & 1 if k.startswith(\'0x\') else int(k[-1])\n\n\ndef tidy(counts):\n    out = {\'0\': 0, \'1\': 0}\n    for k, v in counts.items():\n        out[str(bit(k))] += v\n    return out\n\n\ndef main(backend=None):\n    F_ideal, dec = w.best_decoder(w.choi_state(STEPS, MU))\n    print(f\'Perfect-machine prediction with coupling: {F_ideal:.3f} (0.500 without)\')\n    if backend is None:\n        from qiskit_quantuminspire.qi_provider import QIProvider\n        provider = QIProvider()\n        names = [b.name for b in provider.backends()]\n        name = next((n for n in names if WANT.lower() in n.lower()), None)\n        if name is None:\n            sys.exit(f\'{WANT} not found; machines available: {names}\')\n        backend = provider.get_backend(name)\n    print(\'Using:\', backend.name)\n    s = {\'steps\': STEPS, \'mu\': MU, \'decoder\': dec.tolist()}\n    circs, meta = w.build_batch(s)\n    tc = transpile(circs, backend, optimization_level=3, seed_transpiler=7)\n    print(f\'{len(tc)} circuits, at most {max(sum(1 for i in c.data if i.operation.num_qubits == 2) for c in tc)} two-qubit gates each; {SHOTS} shots each\')\n    counts = []\n    for i in range(0, len(tc), 5):\n        while True:\n            try:\n                job = backend.run(tc[i:i + 5], shots=SHOTS)\n                break\n            except Exception as e:\n                if \'429\' in str(e) or \'maximum\' in str(e):\n                    print(\'  queue full (earlier jobs still waiting), retrying in 60 s...\')\n                    time.sleep(60)\n                else:\n                    raise\n        res = job.result(timeout=None)\n        counts += [tidy(res.get_counts(k)) for k in range(len(res.results))]\n        print(f\'  batch {i // 5 + 1} of {(len(tc) + 4) // 5} done\')\n    summary = w.summarise(meta, counts, f\'\\n{backend.name} (real hardware), shallow circuit:\')\n    json.dump({\'backend\': backend.name, \'circuit\': {\'N\': w.N, \'dt\': w.DT, \'steps\': STEPS, \'mu\': MU},\n               \'prediction_ideal\': F_ideal, \'summary\': summary, \'counts\': counts},\n              open(\'hardware_result_shallow.json\', \'w\'), indent=1)\n    print(\'Saved hardware_result_shallow.json\')\n\n\nif __name__ == \'__main__\':\n    main()\n')
print('Scripts ready')

**2. Log in** (click the link that appears)

In [ ]:
!qi login

**3. Run on Tuna-17 and download the result.** Send the three `mu=` lines (or `hardware_result_shallow.json`) to Claude.

In [ ]:
!python qi_shallow.py Tuna-17
import os
from google.colab import files
if os.path.exists('hardware_result_shallow.json'):
    files.download('hardware_result_shallow.json')